# 02 Ingest — Azure Storage (parquet · full & incremental)

**Doel:** Lees alle actieve `azurestorage`-rijen uit de control table en laad de
bijbehorende parquet-bestanden als Delta-tabellen in `STAGING_AZURESTORAGE`.

## Widgets

| Widget | Default | Doel |
|---|---|---|
| `catalog` | `DEMO` | Unity Catalog catalog name |
| `pipeline_run_id` | `""` | Workflow job run id (ingevuld door Workflow) |
| `mode` | `both` | `full`, `incremental`, of `both` — bepaalt welke `load_type`-rijen verwerkt worden |
| `reset` | `false` | Op `true`: drop doeltabellen + checkpoint folders vóór de run (gebruik bij eerste incremental-run na full→incremental switch) |

## Laadstrategieën (gestuurd door `load_type` in de control table)

| `load_type` | Gedrag |
|---|---|
| `full` | Alle bestanden opnieuw inlezen en de doeltabel volledig overschrijven |
| `incremental` | Auto Loader (`cloudFiles`) — alleen nieuwe bestanden verwerken, checkpoint per doeltabel |

`mode` filtert welke `load_type`-rijen verwerkt worden in deze run. De Workflow-taken `ingest_azurestorage_full` en `ingest_azurestorage_incremental` zetten elk hun eigen mode; `mode=both` is voor ad-hoc handmatige runs vanuit de notebook-UI.

**Switchen tussen full en incrementeel:** Eén `UPDATE` in de control table is genoeg —
geen codewijziging nodig.

```sql
UPDATE DEMO.CONFIG.pipeline_sources
SET    load_type = 'incremental'
WHERE  target_table = 'order_header'
```

⚠️ **Reset bij switch op gevulde tabel:** Wanneer een tabel die al via `full` is geladen wordt omgezet naar `incremental`, dupliceert de eerste incremental-run alle bestaande rijen — Auto Loader heeft geen checkpoint en behandelt elk bronbestand als nieuw. Zet daarom `reset=true` bij die eerste run.

**Checkpoints** (incremental mode): `{source_path}/_checkpoints/{target_table}/`

**Audit-kolommen per doeltabel:**

| Kolom | Bron |
|---|---|
| `_ingestion_timestamp` | `current_timestamp()` |
| `_source_system` | waarde uit control table |
| `_source_file` | `_metadata.file_path` |
| `_last_modified` | `_metadata.file_modification_time` |
| `_pipeline_run_id` | Workflow job run id (widget) |

**Change Data Feed:** elke doeltabel krijgt `delta.enableChangeDataFeed=true` zodat de Silver-laag via `readChangeFeed` + `apply_changes` kan consumeren.

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Catalog")
dbutils.widgets.text("pipeline_run_id", "", "Pipeline Run ID (ingevuld door Workflow)")
dbutils.widgets.dropdown("mode", "both", ["full", "incremental", "both"], "Mode")
dbutils.widgets.dropdown("reset", "false", ["false", "true"], "Reset (drop tabellen + checkpoints)")

catalog          = dbutils.widgets.get("catalog")
pipeline_run_id  = dbutils.widgets.get("pipeline_run_id")
mode             = dbutils.widgets.get("mode")
reset            = dbutils.widgets.get("reset") == "true"

print(f"Catalog          : {catalog}")
print(f"Pipeline run id  : {pipeline_run_id or '(niet opgegeven — handmatig run)'}")
print(f"Mode             : {mode}")
print(f"Reset            : {reset}")

## Stap 1 — Expliciete bron-schema's

Expliciete `StructType`-schema's per `target_table` voorkomen dat Spark Parquet
logical types moet inferen die het niet ondersteunt — met name `INT32 TIME(MILLIS,true)`
(de `[PARQUET_TYPE_ILLEGAL]` error op `SHIFT_START_TIME` / `SHIFT_END_TIME`).

**staging-principe:** deze laag schrijft de data zo dicht mogelijk bij de bron.
TIME-kolommen worden gelezen als `IntegerType` (milliseconden sinds middernacht)
en in die vorm opgeslagen — geen formattering of cast in de ingest-fase. Een column
`COMMENT` documenteert de eenheid voor downstream consumers; conversie naar
`HH:mm:ss` hoort thuis in de Silver/Integration-laag.

Doeltabellen die níet in `TARGET_SCHEMAS` staan vallen terug op schema-inferentie.
Gebruik `staging/schema_inspector.ipynb` om bron-schema's te diagnosticeren.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    DecimalType, DoubleType, IntegerType, StringType, TimestampType,
)

# Column COMMENT voor TIME-kolommen die als IntegerType doorvloeien.
# Delta neemt StructField metadata['comment'] over als column comment in de
# tabeldefinitie zodat downstream consumers de eenheid kunnen aflezen.
TIME_MILLIS_COMMENT = (
    "milliseconds since midnight (source: parquet INT32 TIME(MILLIS,true)); "
    "format to HH:mm:ss in the Silver/Integration layer"
)

# Expliciete bron-schema's per doeltabel.
# IntegerType voor SHIFT_START_TIME / SHIFT_END_TIME omdat parquet ze opslaat
# als INT32 TIME(MILLIS,true) — een logical type dat Spark weigert.
TARGET_SCHEMAS: dict[str, StructType] = {
    "order_header": StructType([
        StructField("ORDER_ID",                DecimalType(38, 0), True),
        StructField("TRUCK_ID",                DecimalType(38, 0), True),
        StructField("LOCATION_ID",             DoubleType(),       True),
        StructField("CUSTOMER_ID",             DecimalType(38, 0), True),
        StructField("DISCOUNT_ID",             StringType(),       True),
        StructField("SHIFT_ID",                DecimalType(38, 0), True),
        StructField("SHIFT_START_TIME",        IntegerType(),      True,
                    metadata={"comment": TIME_MILLIS_COMMENT}),
        StructField("SHIFT_END_TIME",          IntegerType(),      True,
                    metadata={"comment": TIME_MILLIS_COMMENT}),
        StructField("ORDER_CHANNEL",           StringType(),       True),
        StructField("ORDER_TS",                TimestampType(),    True),
        StructField("SERVED_TS",               StringType(),       True),
        StructField("ORDER_CURRENCY",          StringType(),       True),
        StructField("ORDER_AMOUNT",            DecimalType(38, 4), True),
        StructField("ORDER_TAX_AMOUNT",        StringType(),       True),
        StructField("ORDER_DISCOUNT_AMOUNT",   StringType(),       True),
        StructField("ORDER_TOTAL",             DecimalType(38, 4), True),
    ]),
    "order_detail": StructType([
        StructField("ORDER_DETAIL_ID",            DecimalType(38, 0), True),
        StructField("ORDER_ID",                   DecimalType(38, 0), True),
        StructField("MENU_ITEM_ID",               DecimalType(38, 0), True),
        StructField("DISCOUNT_ID",                StringType(),       True),
        StructField("LINE_NUMBER",                DecimalType(38, 0), True),
        StructField("QUANTITY",                   DecimalType(5, 0),  True),
        StructField("UNIT_PRICE",                 DecimalType(38, 4), True),
        StructField("PRICE",                      DecimalType(38, 4), True),
        StructField("ORDER_ITEM_DISCOUNT_AMOUNT", StringType(),       True),
    ]),
}

print(f"Bron-schema's gedefinieerd voor: {list(TARGET_SCHEMAS.keys())}")

## Stap 2 — Control table inlezen

In [ ]:
control_table = f"{catalog}.CONFIG.pipeline_sources"

# mode='both' verwerkt alles; mode='full'/'incremental' filtert op load_type
mode_filter = "" if mode == "both" else f"AND load_type = '{mode}'"

sources_df = spark.sql(
    f"""
    SELECT *
    FROM   {control_table}
    WHERE  source_system = 'azurestorage'
    AND    is_active     = true
    {mode_filter}
    """
)

source_rows = sources_df.collect()
print(f"Actieve azurestorage-bronnen voor mode='{mode}': {len(source_rows)}")
for row in source_rows:
    print(f"  → {row['target_table']} ({row['load_type']}) — {row['file_pattern']}")

if not source_rows:
    print(f"\n(geen rijen matchen mode='{mode}' — no-op)")

## Stap 3 — Ingest per bron

In [ ]:
from pyspark.sql import functions as F

row_count_report = []

for row in source_rows:
    source_path   = row["source_path"]
    file_pattern  = row["file_pattern"]
    target_schema = row["target_schema"]
    target_table  = row["target_table"]
    load_type     = row["load_type"]
    source_system = row["source_system"]

    full_target     = f"{catalog}.{target_schema}.{target_table}"
    checkpoint_path = f"{source_path}/_checkpoints/{target_table}"
    explicit_schema = TARGET_SCHEMAS.get(target_table)
    print(f"\n--- Verwerken: {full_target} (load_type={load_type}) ---")

    # -----------------------------------------------------------------
    # Reset path — drop tabel + checkpoint vóór alles. Idempotent.
    # Gebruik bij eerste incremental-run na een full→incremental switch
    # om duplicaten te voorkomen (Auto Loader heeft geen checkpoint en
    # zou anders alle bestaande bestanden opnieuw als 'nieuw' verwerken).
    # -----------------------------------------------------------------
    if reset:
        print(f"  [reset] Dropping {full_target} en checkpoint {checkpoint_path}")
        spark.sql(f"DROP TABLE IF EXISTS {full_target}")
        try:
            dbutils.fs.rm(checkpoint_path, recurse=True)
        except Exception as exc:
            print(f"  [reset] Checkpoint pad bestond niet of kon niet verwijderd worden: {exc}")

    if explicit_schema is not None:
        print(f"  Bron-schema       : expliciet ({len(explicit_schema)} kolommen)")
    else:
        print("  Bron-schema       : geïnferred door Spark")

    if load_type == "full":
        # -----------------------------------------------------------------
        # FULL LOAD — lees alle bestanden die overeenkomen met het glob-filter
        # en overschrijf de doeltabel volledig.
        # -----------------------------------------------------------------
        reader = spark.read.format("parquet")
        if explicit_schema is not None:
            reader = reader.schema(explicit_schema)

        raw_df = (
            reader
            .option("pathGlobFilter", file_pattern)
            .option("recursiveFileLookup", "false")
            .load(source_path)
        )

        # Voeg de vijf audit-kolommen toe.
        enriched_df = (
            raw_df
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            .drop("_metadata")
        )

        # Schrijf als Delta (full = overschrijven).
        # delta.enableChangeDataFeed wordt direct op de writer gezet zodat
        # version 0 al CDF-events bevat — Silver leest vanaf startingVersion=0.
        (
            enriched_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.enableChangeDataFeed", "true")
            .saveAsTable(full_target)
        )

        written_count = spark.table(full_target).count()
        row_count_report.append((full_target, "full", written_count))
        print(f"  [full] Geschreven: {written_count:,} rijen → {full_target}")

    elif load_type == "incremental":
        # -----------------------------------------------------------------
        # INCREMENTAL LOAD — Auto Loader (cloudFiles) met checkpoint per
        # doeltabel.  trigger(availableNow=True) verwerkt alle nieuwe
        # bestanden en stopt — geschikt voor batch Workflows.
        # -----------------------------------------------------------------
        print(f"  Checkpoint pad   : {checkpoint_path}")
        print(f"  Bestandsfilter   : {file_pattern}")

        rows_before = (
            spark.table(full_target).count()
            if spark.catalog.tableExists(full_target)
            else 0
        )

        reader = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format",         "parquet")
            .option("cloudFiles.schemaLocation", checkpoint_path)
            .option("pathGlobFilter",            file_pattern)
            .option("recursiveFileLookup",       "false")
        )
        if explicit_schema is not None:
            reader = reader.schema(explicit_schema)

        # delta.enableChangeDataFeed wordt direct op de writeStream gezet zodat
        # version 0 al CDF-events bevat — Silver leest vanaf startingVersion=0.
        stream_query = (
            reader
            .load(source_path)
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            .drop("_metadata")
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema",         "true")
            .option("delta.enableChangeDataFeed", "true")
            .trigger(availableNow=True)
            .toTable(full_target)
        )

        stream_query.awaitTermination()

        rows_after  = spark.table(full_target).count()
        delta_count = rows_after - rows_before
        row_count_report.append((full_target, "incremental", delta_count))
        print(f"  [incremental] Nieuwe rijen: {delta_count:,} | Totaal: {rows_after:,} → {full_target}")

    else:
        raise ValueError(
            f"Onbekend load_type='{load_type}' voor tabel '{target_table}'. "
            "Verwacht 'full' of 'incremental'."
        )

    # -----------------------------------------------------------------
    # Change Data Feed — idempotente vangnetregel voor het geval een eerdere
    # ingest-run de tabel nog zonder CDF heeft gemaakt. Voor nieuwe tabellen
    # is dit een no-op (de writer-option hierboven heeft het al gezet).
    # -----------------------------------------------------------------
    spark.sql(
        f"ALTER TABLE {full_target} "
        f"SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
    )
    print(f"  CDF              : delta.enableChangeDataFeed=true (geverifieerd)")

## Stap 4 — Validatie & row counts

In [ ]:
print("\n=== Row count samenvatting ===")
for table, mode, count in row_count_report:
    label = "geschreven" if mode == "full" else "nieuwe rijen"
    print(f"  [{mode}] {table}: {count:,} {label}")

## Resultaat

Alle actieve `azurestorage`-bronnen die matchen op `mode` zijn ingeladen als Delta-tabellen in `STAGING_AZURESTORAGE`. Elke tabel bevat de vijf audit-kolommen (`_ingestion_timestamp`, `_source_system`, `_source_file`, `_last_modified`, `_pipeline_run_id`) en heeft `delta.enableChangeDataFeed=true` zodat Silver via `readChangeFeed` + `apply_changes` kan consumeren.

**Full mode** (`mode=full`): rijen met `load_type='full'` — alle bestanden opnieuw verwerkt, doeltabel volledig overschreven.

**Incremental mode** (`mode=incremental`): rijen met `load_type='incremental'` — alleen nieuwe bestanden via Auto Loader (`cloudFiles`). Checkpoints onder `_checkpoints/{target_table}/` in het external volume. Herdraaien levert uitsluitend de delta — geen dubbele rijen.

**Both mode** (`mode=both`): verwerkt alle actieve rijen ongeacht `load_type` — handig voor ad-hoc handmatige runs.

**Reset** (`reset=true`): drop doeltabellen en checkpoint-folders vóór de run. Gebruik bij de eerste incremental-run na een full→incremental switch op een gevulde tabel — anders dupliceert Auto Loader bij gebrek aan checkpoint alle bestaande bestanden.

**Schema-aanpak (staging):** Doeltabellen in `TARGET_SCHEMAS` worden gelezen met een expliciet `StructType`. TIME-kolommen (`SHIFT_START_TIME`, `SHIFT_END_TIME`) worden als `IntegerType` ingelezen en als zodanig opgeslagen — milliseconden sinds middernacht. Een column `COMMENT` op die kolommen documenteert de eenheid. De conversie naar `'HH:mm:ss'` hoort thuis in de Silver/Integration-laag, niet in deze staging-ingest.

**Demo-tip:** Verander `load_type` in de control table met één `UPDATE`, draai het Workflow opnieuw — de notebook past zijn gedrag automatisch aan zonder codewijziging. Voor de allereerste incremental-run na zo'n switch: zet `reset=true`.